In [ ]:
import sys
sys.path.insert(0, "/users/eleves-a/2023/roman.lendormy/reinforcement_learning/Project/RL_project/src")

In [ ]:
from mct.train import TrainConfig, run

In [ ]:
# cfg = TrainConfig(
#     steps            = 100_000_000,
#     load             = "/Data/roman.lendormy/rl_checkpoints_2/ckpt_42516k.pt",
#     n_envs           = 8,
#     lr               = 5e-5, #ex 3e-4
#     gamma            = 0.995,
#     ent_coef         = 0.03,
#     vf_coef          = 0.25,
#     clip_eps         = 0.1,
#     epochs           = 8,
#     n_steps          = 1024, #ex 512
#     batch            = 4096, #ex 2048
#     gae_lambda       = 0.99, #ex 0.98
#     checkpoint_every = 500_000,
#     checkpoint_dir   = "/Data/roman.lendormy/rl_checkpoints_2",
#     save             = "/Data/roman.lendormy/rl_checkpoints_2/ppo_blockblast.pt",
#     plot             = "/Data/roman.lendormy/rl_checkpoints_2/training_curves.png",
# )
# trainer, history = run(cfg)

# trainer.save("/Data/roman.lendormy/rl_checkpoints_2/ppo_blockblast_final.pt")

In [ ]:
import sys
import torch
sys.path.insert(0, "/users/eleves-a/2023/roman.lendormy/reinforcement_learning/Project/RL_project/src")

from mct.train import TrainConfig, run
from mct.ppo_agent import PPOTrainer
from blockblast.block_blast_3p_env import BlockBlast3PEnv

# Recréer un trainer vide puis charger le checkpoint
envs = [BlockBlast3PEnv() for _ in range(1)]  # 1 env factice juste pour init
trainer = PPOTrainer(envs=envs, device=torch.device("cpu"))
trainer.load("/Data/roman.lendormy/rl_checkpoints_2/ckpt_42516k.pt")

In [ ]:
# C'est tout ce qu'il faut — pas de training MCTS
from mct.mcts_agent import compare_ppo_vs_mcts
from blockblast.block_blast_3p_env import BlockBlast3PEnv


In [ ]:

# # trainer est déjà chargé avec ton checkpoint 42.5M steps
# results = compare_ppo_vs_mcts(
#     model      = trainer.model,
#     env_fn     = lambda: BlockBlast3PEnv(),
#     device     = torch.device("cpu"),
#     n_episodes = 100,
# )

In [ ]:
from mct.mcts_agent import MCTSAgent
import torch

agent = MCTSAgent(
    model   = trainer.model,
    device  = torch.device("cpu"),
    gamma   = 0.99,
    verbose = True,  # affiche le score et le temps par round
)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

env = BlockBlast3PEnv(render_mode="human")
obs, info = env.reset()

STEPS = 200
total_reward = 0

%matplotlib inline

for step_idx in range(STEPS):
    # Dessiner directement via _draw_figure sans passer par render()
    env._draw_figure()
    
    clear_output(wait=True)
    display(env.fig)
    plt.close(env.fig)
    env.fig = None  # force recréation au prochain step
    
    print(f"Step {step_idx+1} | Combo: {env.combo} | Pieces used: {env.pieces_used.tolist()}")
    print(f"Total reward: {total_reward:.1f}")

    action = agent.select_action(env)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    time.sleep(0.3)

    if terminated or truncated:
        env._draw_figure()
        clear_output(wait=True)
        display(env.fig)
        plt.close(env.fig)
        env.fig = None
        print(f"\nGame Over après {step_idx+1} steps | Reward finale : {total_reward:.1f}")
        break

env.close()

# Teaching avec MCTS

## Collecte

In [ ]:
trainer.load("/Data/roman.lendormy/rl_checkpoints_2/ckpt_42516k.pt")
trainer.model = trainer.model.to(torch.device("cuda"))

In [ ]:
from mct.mcts_collect import collect_mcts_dataset

collect_mcts_dataset(
    trainer    = trainer,
    env_fn     = lambda: BlockBlast3PEnv(),
    n_episodes = 1000,
    save_path  = "/Data/roman.lendormy/rl_checkpoints_2/mcts_dataset_v2.npz",
    device     = torch.device("cuda"),
)

In [ ]:
# from mct.ppo_finetune import ppo_finetune_on_mcts

# history = ppo_finetune_on_mcts(
#     trainer      = trainer,
#     dataset_path = "/Data/roman.lendormy/rl_checkpoints_2/mcts_dataset_v2.npz",
#     save_path    = "/Data/roman.lendormy/rl_checkpoints_2/ppo_mcts_ft_v3.pt",
#     device       = torch.device("cuda"),
#     n_epochs     = 5,
#     lr           = 5e-6,    # 10x plus bas
#     clip_eps     = 0.05,    # clipping très serré
#     vf_coef      = 0.1,     # réduire le poids de la VF loss
#     plot_path    = "/Data/roman.lendormy/rl_checkpoints_2/ft_curves_v3.png",
# )

In [ ]:
import numpy as np
data = np.load("/Data/roman.lendormy/rl_checkpoints_2/mcts_dataset_v2.npz")
print("Rewards min/max/mean:", data["rewards"].min(), data["rewards"].max(), data["rewards"].mean())
print("Dones sum:", data["dones"].sum(), "/ episodes")

In [ ]:
from mct.mcts_agent import MCTSAgent
import torch

trainer.model = trainer.model.to(torch.device("cuda"))

agent = MCTSAgent(
    model   = trainer.model,
    device  = torch.device("cuda"),
    gamma   = 0.99,
    verbose = False,)

# PPO greedy
print("--- PPO greedy ---")
ppo_stats = agent.evaluate(
    lambda: BlockBlast3PEnv(),
    n_episodes = 50,
    use_mcts   = False,
)

# MCTS triplet
print("\n--- MCTS triplet ---")
mcts_stats = agent.evaluate(
    lambda: BlockBlast3PEnv(),
    n_episodes = 50,
    use_mcts   = True,
)

# Comparaison
delta = mcts_stats["mean_return"] - ppo_stats["mean_return"]
print(f"\n=== Comparison ===")
print(f"  Return gain   : {delta:+.2f} ({delta/max(abs(ppo_stats['mean_return']),1)*100:+.1f}%)")
print(f"  Length gain   : {mcts_stats['mean_length'] - ppo_stats['mean_length']:+.1f} steps")
print(f"  MCTS overhead : {mcts_stats['mean_time_per_round_ms']:.1f} ms / round")

In [ ]:
import numpy as np

data = np.load("/Data/roman.lendormy/rl_checkpoints_2/mcts_dataset_v2.npz")

# Reconstituer les rewards 3-coups par round
# Dans le dataset, chaque épisode = 35 steps ~= 11-12 rounds de 3 coups
rewards = data["rewards"]
dones   = data["dones"]

# Simuler les cumulative_reward_3steps manuellement
# (grouper par triplets consécutifs dans un épisode)
gamma = 0.99
r3_list = []

i = 0
while i + 2 < len(rewards):
    # vérifier qu'on ne traverse pas une frontière d'épisode
    if dones[i] == 0 and dones[i+1] == 0:
        r3 = rewards[i] + gamma * rewards[i+1] + gamma**2 * rewards[i+2]
        r3_list.append(r3)
        i += 3
    else:
        i += 1

r3 = np.array(r3_list)
print(f"Cumulative 3-step rewards:")
print(f"  min    : {r3.min():.2f}")
print(f"  max    : {r3.max():.2f}")
print(f"  mean   : {r3.mean():.2f}")
print(f"  median : {np.median(r3):.2f}")
print(f"  p75    : {np.percentile(r3, 75):.2f}")
print(f"  p95    : {np.percentile(r3, 95):.2f}")
print(f"\nsymlog de ces valeurs:")
print(f"  min    : {np.sign(r3.min()) * np.log1p(abs(r3.min())):.2f}")
print(f"  max    : {np.sign(r3.max()) * np.log1p(abs(r3.max())):.2f}")
print(f"  mean   : {np.sign(r3.mean()) * np.log1p(abs(r3.mean())):.2f}")

print(f"\nValeur brute PPO (symlog space) : ~14.18")
print(f"Valeur brute PPO (symexp) : ~1.4M")

In [ ]:
# ===== Value weight sweep =====

from mct.value_weight_sweep import run_sweep
from blockblast.block_blast_3p_env import BlockBlast3PEnv
import torch

# S'assurer que le bon checkpoint est chargé
trainer.load("/Data/roman.lendormy/rl_checkpoints_2/ckpt_42516k.pt")
trainer.model = trainer.model.to(torch.device("cuda"))

results = run_sweep(
    model      = trainer.model,
    env_fn     = lambda: BlockBlast3PEnv(),
    device     = torch.device("cuda"),
    n_episodes = 50,
    weights    = [0.0, 0.05, 0.1, 0.3, 0.5, 1.0],
    plot_path  = "/Data/roman.lendormy/rl_checkpoints_2/value_weight_sweep.png",
)

In [ ]:
from mct.mcts_agent import compare_ppo_vs_mcts

results = compare_ppo_vs_mcts(
    model      = trainer.model,
    env_fn     = lambda: BlockBlast3PEnv(),
    device     = torch.device("cuda"),
    n_episodes = 50,
)

# PPO online with MCTS

In [ ]:
from mct.mcts_ppo_trainer import MCTSPPOTrainer
from blockblast.block_blast_3p_env import BlockBlast3PEnv
import torch, os

os.makedirs("/Data/roman.lendormy/rl_checkpoints_2/mcts_ppo", exist_ok=True)

# 4 envs suffit — le goulot d'étranglement est MCTS pas le GPU
envs = [BlockBlast3PEnv() for _ in range(4)]

mcts_trainer = MCTSPPOTrainer(
    envs       = envs,
    device     = torch.device("cuda"),
    lr         = 1e-4,
    n_steps    = 128,
    n_epochs   = 4,
    batch_size = 256,
    ent_coef   = 0.01,
    vf_coef    = 0.25,
    gae_lambda = 0.98,
    gamma      = 0.99,
)

# Charger le checkpoint original comme point de départ
mcts_trainer.load("/Data/roman.lendormy/rl_checkpoints_2/ckpt_42516k.pt")
mcts_trainer.model = mcts_trainer.model.to(torch.device("cuda"))

In [ ]:
history = mcts_trainer.train(
    total_timesteps  = 5_000_000,
    log_interval     = 1,
    checkpoint_every = 500_000,
    checkpoint_dir   = "/Data/roman.lendormy/rl_checkpoints_2/mcts_ppo",
)